# Tarea 2 · Voces y documentos peruanos: del archivo escaneado al texto buscable
## Parte 1 — Corpus y extracción (OCR + ASR)

**Autora:** Mirian de la Cruz Condori · **Curso:** Diplomado de Python · **Notebook:** `01_extraccion.ipynb`

> **Pregunta guía:** ¿Podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

**Objetivo de esta parte.** Construir un corpus verificable de **25 documentos** peruanos (8 impresos, 8 fotografías con dificultad visual y 9 manuscritos) y **30 clips de audio** en español peruano (36,7 min reales), y extraer su texto con un pipeline reproducible: OCR con **PaddleOCR**, ASR con **faster-whisper** (Whisper *small*) con **VAD**, **diarización** de hablantes y **seudonimización** previa a cualquier publicación. La evaluación con referencias humanas (CER/WER) se realiza en `02_evaluacion.ipynb`.

### Conceptos clave
| Término | Qué es | Cómo se usa aquí |
|---|---|---|
| **OCR** (*Optical Character Recognition*) | Convierte la imagen de un texto en caracteres. Primero **detecta** las líneas y luego las **reconoce**. | PaddleOCR 3.7 (modelos PP-OCRv6 *medium*) sobre los 25 documentos. |
| **ASR** (*Automatic Speech Recognition*) | Convierte la voz en texto. Whisper es un modelo codificador-decodificador entrenado en 680 000 h de audio multilingüe. | faster-whisper 1.2 (Whisper *small*, int8, CPU), idioma `es`. |
| **VAD** (*Voice Activity Detection*) | Detecta los tramos con voz y descarta silencios o ruido antes del ASR, lo que reduce las alucinaciones. | Silero VAD integrado en faster-whisper (`vad_filter=True`). |
| **Diarización** | Responde a la pregunta «¿quién habló cuándo?»: segmenta el audio por hablante sin saber quiénes son. | sherpa-onnx: segmentación pyannote 3.0 + embeddings WeSpeaker + *clustering* automático. |
| **Seudonimización** | Reemplaza datos personales por marcadores (`[NOMBRE_1]`, `[DNI]`…) y guarda el mapa de correspondencias en un lugar privado. | Expresiones regulares y NER de spaCy, seguidas de **revisión humana**. |

### Pipeline
1. **Corpus** (scripts `00`–`05`): copia verificada de originales → fuentes y licencias por SHA-1 en Wikimedia Commons → normalización de audio con FFmpeg → 30 clips WAV de 16 kHz en mono → clasificación documental.
2. **Extracción** (este notebook): OCR → ASR + VAD → diarización → seudonimización.
3. **Control de calidad**: hashes, duplicados exactos y casi duplicados, archivos vacíos o corruptos, audio sin voz y registro de descartes.
4. **Exportación**: `datos/documentos.csv`, `datos/transcripciones.csv` y `datos/fuentes.csv` (solo con texto seudonimizado).
5. **EDA**: gráficos Plotly y extremos del corpus.

Los procesos pesados guardan una **caché por elemento** en `datos/raw/cache/` (no versionada). Con `FORZAR = True` se recalcula todo desde cero con las mismas funciones (`scripts/ocr.py`, `scripts/asr.py`, `scripts/diarizacion.py`, `scripts/privacidad.py`).

### Convención de integridad
Todo valor proviene de una ejecución real sobre los archivos del corpus. Lo que requiere juicio humano (referencias, número final de hablantes, idioma confirmado por escucha, fuentes faltantes) se marca como **`PENDIENTE_REVISION_HUMANA`** y no se rellena automáticamente. La revisión humana de privacidad de los textos seudonimizados la realizó la autora el 2026-10-25 (`revision_privacidad = REVISADO_HUMANAMENTE`).

## 0. Configuración del entorno

In [1]:
import sys, os, json, platform, warnings, subprocess
from datetime import date, datetime
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
warnings.filterwarnings("ignore")

# IMPORTANTE (Windows): faster-whisper/ctranslate2 debe importarse ANTES que PaddleOCR.
# En el orden inverso ctranslate2 falla con "WinError 127" (conflicto de carga de DLL).
import asr            # faster-whisper
import diarizacion    # sherpa-onnx
import ocr            # PaddleOCR (paddle se importa de forma diferida) + Tesseract
import privacidad     # spaCy + regex
import graficos       # exportación PNG opcional con límite de tiempo + cronómetro por sección
from utilidades import RAW, REPORTES, DATOS, binario, ffprobe, sha256

import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

import logging
for _nombre in ["kaleido", "choreographer", "logistro", "browser_async", "chromium"]:
    logging.getLogger(_nombre).setLevel(logging.ERROR)  # la exportación PNG no debe ensuciar las salidas
logging.getLogger().setLevel(logging.WARNING)

FORZAR = False          # True = recalcular OCR, ASR y diarización ignorando la caché
EXPORTAR_PNG = False    # True = exportar PNG estáticos (Kaleido) en subproceso con límite de 60 s por figura
PENDIENTE = "PENDIENTE_REVISION_HUMANA"
# Revisión humana de privacidad: la autora revisó los textos seudonimizados (pestañas Documentos y Audios
# del dashboard) y la dio por realizada el 2026-10-25.
REVISION_PRIVACIDAD = "REVISADO_HUMANAMENTE"
crono = graficos.Cronometro()
crono.marca("0. Configuración e importaciones")
pd.set_option("display.max_colwidth", 120, "display.width", 200)
print("Ejecución del notebook:", datetime.now().strftime("%Y-%m-%d %H:%M"))

Ejecución del notebook: 2026-10-25 16:32


In [2]:
import importlib.metadata as md_
import faster_whisper, ctranslate2, sherpa_onnx, spacy, pytesseract
version = lambda p: md_.version(p)
ffmpeg_v = subprocess.run([binario("ffmpeg"), "-version"], capture_output=True, text=True).stdout.split("\n")[0]
pytesseract.pytesseract.tesseract_cmd = str(ocr.TESSERACT_EXE)
import psutil
versiones = pd.DataFrame([
    ("Sistema", f"{platform.system()} {platform.release()} ({platform.version()})"),
    ("CPU / RAM", f"{platform.processor()} · {os.cpu_count()} hilos · {psutil.virtual_memory().total/1024**3:.1f} GB"),
    ("Python", platform.python_version()),
    ("PaddleOCR / PaddlePaddle", f"{version('paddleocr')} / {version('paddlepaddle')} (CPU) · {ocr.MODELOS_PADDLE}"),
    ("faster-whisper / CTranslate2", f"{faster_whisper.__version__} / {ctranslate2.__version__} · whisper-{asr.MODELO} ({asr.TIPO_COMPUTO})"),
    ("sherpa-onnx (diarización)", sherpa_onnx.__version__),
    ("spaCy / modelo NER", f"{spacy.__version__} / es_core_news_md {version('es_core_news_md')}"),
    ("Tesseract (OCR alternativo, Parte 2)", f"{pytesseract.get_tesseract_version()} · spa tessdata_best 4.1.0"),
    ("FFmpeg", ffmpeg_v.replace("ffmpeg version ", "")),
    ("pandas / NumPy / Plotly", f"{pd.__version__} / {np.__version__} / {plotly.__version__}"),
], columns=["Componente", "Versión"])
display(versiones.style.hide(axis="index"))

Componente,Versión
Sistema,Windows 11 (10.0.26200)
CPU / RAM,"Intel64 Family 6 Model 170 Stepping 4, GenuineIntel · 18 hilos · 15.5 GB"
Python,3.13.7
PaddleOCR / PaddlePaddle,3.7.0 / 3.3.1 (CPU) · PP-OCRv6_medium_det + PP-OCRv6_medium_rec
faster-whisper / CTranslate2,1.2.1 / 4.8.3 · whisper-small (int8)
sherpa-onnx (diarización),2.0.0
spaCy / modelo NER,3.8.16 / es_core_news_md 3.8.0
"Tesseract (OCR alternativo, Parte 2)",5.4.0.20240606 · spa tessdata_best 4.1.0
FFmpeg,9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
pandas / NumPy / Plotly,3.1.0 / 2.3.5 / 7.1.0


In [3]:
# Tema visual de los gráficos: superficie clara, tinta neutra, cuadrícula fina y paleta categórica
# validada para daltonismo (las tres primeras posiciones son distinguibles en todos los pares).
TINTA, TINTA2, TENUE, GRID, EJE, SUPERFICIE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
CATEGORICA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
COLOR_TIPO = {"impreso": CATEGORICA[0], "foto": CATEGORICA[1], "manuscrito": CATEGORICA[2]}
ORDEN_TIPO = ["impreso", "foto", "manuscrito"]
FUENTE_TXT = "Segoe UI, system-ui, -apple-system, sans-serif"

pio.templates["tarea2"] = go.layout.Template(layout=dict(
    font=dict(family=FUENTE_TXT, size=13, color=TINTA2),
    paper_bgcolor=SUPERFICIE, plot_bgcolor=SUPERFICIE, colorway=CATEGORICA,
    title=dict(x=0.01, xanchor="left", font=dict(size=18, color=TINTA)),
    xaxis=dict(gridcolor=GRID, linecolor=EJE, zeroline=False, ticks="", tickfont=dict(color=TENUE)),
    yaxis=dict(gridcolor=GRID, linecolor=EJE, zeroline=False, ticks="", tickfont=dict(color=TINTA2)),
    hoverlabel=dict(bgcolor="white", bordercolor=EJE, font=dict(family=FUENTE_TXT, color=TINTA, size=12)),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0, font=dict(color=TINTA2)),
    margin=dict(l=20, r=30, t=100, b=50), bargap=0.35,
))
pio.templates.default = "plotly_white+tarea2"
# Salida interactiva (Jupyter, VS Code, nbviewer). La imagen PNG NO se genera aquí: con Kaleido tardaba
# de 3 s a más de 6 min por figura. Ver la celda opcional de exportación (EXPORTAR_PNG).
pio.renderers.default = "plotly_mimetype+notebook_connected"
FIGURAS = {}  # figuras para la exportación opcional

## 1. Carga y validación del corpus
Los scripts `00`–`05` generaron los reportes de `datos/reportes/`. Aquí se **re-verifica en vivo** que cada archivo de `datos/raw/` exista, que su hash coincida con el registrado y que se cumplan los requisitos.

In [4]:
crono.marca("1. Carga y validación del corpus (hashes y ffprobe en vivo)")
integridad = pd.read_csv(REPORTES / "integridad_archivos.csv")
clasif = pd.read_csv(REPORTES / "clasificacion_documentos.csv")
f_docs = pd.read_csv(REPORTES / "fuentes_documentos_commons.csv")
f_audio = pd.read_csv(REPORTES / "fuentes_audio_commons.csv")
val_audio = pd.read_csv(REPORTES / "validacion_audio_originales.csv")
clips = pd.read_csv(REPORTES / "clips_audio.csv")

docs_int = integridad[integridad["modalidad"] == "documento"].set_index("archivo")
hash_doc_ok = all(sha256(RAW / "documentos" / a) == h for a, h in docs_int["sha256"].items())
hash_clip_ok = all(sha256(RAW / "audio" / "clips" / a) == h for a, h in zip(clips["archivo"], clips["sha256"]))
dur_vivo = {a: float(ffprobe(RAW / "audio" / "clips" / a)["format"]["duration"]) for a in clips["archivo"]}
min_reales = sum(dur_vivo.values()) / 60
conteo = clasif["tipo"].value_counts()

checks = [
    ("25 documentos", len(clasif) == 25, f"{len(clasif)} documentos"),
    ("8 impresos / 8 fotos / 9 manuscritos",
     conteo.to_dict() == {"manuscrito": 9, "impreso": 8, "foto": 8},
     f"impreso {conteo.get('impreso', 0)} · foto {conteo.get('foto', 0)} · manuscrito {conteo.get('manuscrito', 0)}"),
    ("Documentos legibles, sin duplicados exactos",
     docs_int["legible"].all() and (docs_int["duplicado_de"].fillna("") == "").all(), "verificación PIL + SHA-256"),
    ("Hash de documentos igual al registrado", hash_doc_ok, "SHA-256 recalculado ahora"),
    ("≥ 20 clips válidos", int(clips["valido"].sum()) >= 20, f"{int(clips['valido'].sum())} clips válidos"),
    ("≥ 30 min reales", min_reales >= 30, f"{min_reales:.2f} min (ffprobe en vivo)"),
    ("Clips sin duplicados ni silencio", not clips["duplicado"].any() and not clips["silencio"].any(), "SHA-256 + RMS"),
    ("Hash de clips igual al registrado", hash_clip_ok, "SHA-256 recalculado ahora"),
    ("Fuentes de audio verificadas (SHA-1 Commons)", bool(f_audio["sha1_coincide"].all()), f"{len(f_audio)} fuentes"),
]
validacion = pd.DataFrame(checks, columns=["Requisito", "Cumple", "Evidencia"])
validacion["Cumple"] = validacion["Cumple"].map({True: "✅", False: "❌"})
display(validacion.style.hide(axis="index"))
assert (validacion["Cumple"] == "✅").all(), "Hay requisitos del corpus sin cumplir"

Requisito,Cumple,Evidencia
25 documentos,✅,25 documentos
8 impresos / 8 fotos / 9 manuscritos,✅,impreso 8 · foto 8 · manuscrito 9
"Documentos legibles, sin duplicados exactos",✅,verificación PIL + SHA-256
Hash de documentos igual al registrado,✅,SHA-256 recalculado ahora
≥ 20 clips válidos,✅,30 clips válidos
≥ 30 min reales,✅,36.72 min (ffprobe en vivo)
Clips sin duplicados ni silencio,✅,SHA-256 + RMS
Hash de clips igual al registrado,✅,SHA-256 recalculado ahora
Fuentes de audio verificadas (SHA-1 Commons),✅,7 fuentes


In [5]:
# Composición del corpus de audio por fuente (las etiquetas evitan nombres de personas)
ETIQUETA_FUENTE = {
    "audio_A": "A · Entrevista en Jr. de la Unión",
    "audio_B": "B · Entrevista en Plaza San Martín (1)",
    "audio_C": "C · Entrevista en Plaza San Martín (2)",
    "audio_D": "D · Excombatiente del Cenepa",
    "audio_E": "E · Entrevistas en ceremonia patronal",
    "audio_F": "F · Conferencia de prensa (Plaza San Martín)",
    "audio_G": "G · Vendedoras en El Agustino",
}
clips["fuente_id"] = clips["fuente"].str.replace(r"\.\w+$", "", regex=True)
clips["fuente_etiqueta"] = clips["fuente_id"].map(ETIQUETA_FUENTE)
clips["duracion_vivo_s"] = clips["archivo"].map(dur_vivo)
resumen_audio = (clips.groupby(["fuente_id", "fuente_etiqueta"])
                 .agg(clips=("id", "count"), minutos=("duracion_vivo_s", lambda s: round(s.sum() / 60, 2)))
                 .reset_index()
                 .merge(f_audio.assign(fuente_id=f_audio["archivo_local"].str.replace(r"\.\w+$", "", regex=True))
                        [["fuente_id", "licencia", "autor"]], on="fuente_id"))
display(resumen_audio.drop(columns="fuente_id").style.hide(axis="index"))
print(f"TOTAL: {len(clips)} clips · {clips['duracion_vivo_s'].sum() / 60:.2f} min reales · {clips['fuente_id'].nunique()} fuentes")

fuente_etiqueta,clips,minutos,licencia,autor
A · Entrevista en Jr. de la Unión,7,8.180000,CC BY-SA 4.0,Jean Franco Toribio
B · Entrevista en Plaza San Martín (1),3,3.340000,CC BY-SA 4.0,Jean Franco Toribio
C · Entrevista en Plaza San Martín (2),2,2.840000,CC BY-SA 4.0,Jean Franco Toribio
D · Excombatiente del Cenepa,4,5.470000,CC BY-SA 4.0,Jean Franco Toribio
E · Entrevistas en ceremonia patronal,4,5.440000,CC0,Sebastian Enrique Tanca Huatuco
F · Conferencia de prensa (Plaza San Martín),7,8.290000,CC BY-SA 4.0,QM Keen
G · Vendedoras en El Agustino,3,3.150000,CC BY-SA 4.0,JorgeFlores-pe


TOTAL: 30 clips · 36.72 min reales · 7 fuentes


### Fuentes y licencias
Todas las fuentes identificadas provienen de **Wikimedia Commons**. La correspondencia entre cada archivo local y su página de Commons se **demostró por SHA-1** (archivo idéntico byte a byte); no se asumió por el nombre. Las fuentes que no se pudieron verificar quedan como `PENDIENTE_REVISION_HUMANA` y **no se inventan**.

In [6]:
lic = pd.concat([
    f_docs.assign(modalidad="documento")[["modalidad", "licencia"]],
    f_audio.assign(modalidad="audio")[["modalidad", "licencia"]],
])
display(lic.value_counts().rename("archivos").reset_index().style.hide(axis="index"))
pend = f_docs.loc[f_docs["licencia"] == PENDIENTE, "archivo"].tolist()
print(f"Documentos con fuente verificada: {len(f_docs) - len(pend)}/{len(f_docs)} · pendientes: {', '.join(pend)}")

modalidad,licencia,archivos
documento,Public domain,10
documento,PENDIENTE_REVISION_HUMANA,6
audio,CC BY-SA 4.0,6
documento,CC BY-SA 4.0,5
documento,CC BY 4.0,2
documento,CC BY 2.0,1
documento,CC0,1
audio,CC0,1


Documentos con fuente verificada: 19/25 · pendientes: doc_005.png, doc_006.png, doc_007.png, doc_009.png, doc_010.png, doc_021.jpg


### Incidencias metodológicas detectadas al construir el corpus
1. **Clips originales defectuosos.** 19 de los 20 clips iniciales duraban entre 3 y 13 ms y eran duplicados exactos entre sí. Se **descartaron** con registro y se regeneraron.
2. **Duración sobrestimada en Commons.** Los WebM A–D (grabados con VLC) tienen timestamps que empiezan entre 18 y 22 h de *offset* y no declaran su duración. Commons (getID3) la estima como `tamaño × 8 / 128 kbps` (95,7 min en total), pero el ~78 % de los bytes es video VP8. La duración **real** del audio, decodificado completo con FFmpeg, era de 25,28 min. Por eso se añadieron dos fuentes (F y G), verificadas por SHA-1 y ffprobe.
3. **Fuentes descartadas por idioma.** La serie de Shansha (Áncash) en Commons está declarada en **quechua huaylas**, así que se excluyó por no cumplir el requisito de español.
4. **PaddlePaddle 3.3.1 con oneDNN** falla en CPU (`ConvertPirAttribute2RuntimeAttribute`). Se desactivó oneDNN (`enable_mkldnn=False`) en lugar de cambiar versiones. El costo es un OCR más lento.
5. **Conflicto de DLL en Windows** entre paddle y ctranslate2: se resuelve con el orden de importación indicado en la configuración.
6. **faster-whisper 1.2.1 con PyAV 19.** El primer intento de ASR falló en los 30 clips (`TypeError: open() got an unexpected keyword argument 'metadata_errors'`). Como los clips ya son WAV de 16 kHz en mono, se leen con `soundfile`. Los registros del intento fallido se conservan en `datos/raw/cache/asr_faster_whisper_small/intento_fallido_pyav19/`.
7. **Exportación PNG de Plotly (Kaleido).** Dentro del kernel de Jupyter tardaba entre 3 s y más de 6 min por figura; la primera ejecución de este notebook tomó 12,2 min por esa causa. Se retiró del flujo obligatorio y quedó como exportación opcional en un subproceso con límite de tiempo, donde tarda unos 4 s por figura.

## 2. OCR principal con PaddleOCR
**Configuración:**
- `lang="es"`, que selecciona los modelos **PP-OCRv6_medium_det** y **PP-OCRv6_medium_rec**.
- Sin módulos geométricos opcionales: `use_doc_orientation_classify=False`, `use_doc_unwarping=False`, `use_textline_orientation=False`.
- `enable_mkldnn=False`, por la incidencia 4.
- Detección con los parámetros por defecto: `limit_side_len=64` (`min`), `max_side_limit=4000`, `thresh=0.3`, `box_thresh=0.6` y `unclip_ratio=1.5`.

**Preprocesamiento:** no se aplica *resize*, *deskew*, rotación, contraste ni escala de grises. La única conversión es RGBA → RGB sobre fondo blanco cuando la imagen tiene canal alfa. El detector reduce internamente las imágenes con un lado mayor de 4000 px, pero el reconocimiento trabaja sobre la imagen original. Los originales nunca se modifican.

**Texto:** las líneas se ordenan como las devuelve PaddleOCR (de arriba abajo y de izquierda a derecha) y se unen con saltos de línea. El **tiempo** mide únicamente `predict()` por documento, en CPU y sin otros procesos pesados en paralelo.

In [7]:
crono.marca("2. OCR PaddleOCR (lectura de caché si existe)")
documentos = sorted((RAW / "documentos").iterdir())
pendientes = [p for p in documentos if FORZAR or not (ocr.CACHE_PADDLE / f"{p.stem}.json").exists()]
motor = None
if pendientes:
    motor, t_carga = ocr.crear_paddle()
    print(f"PaddleOCR cargado en {t_carga:.1f} s; documentos por procesar: {len(pendientes)}")
res_ocr, errores_ocr = [], []
for p in documentos:
    try:
        res_ocr.append(ocr.ocr_paddle(motor, p, forzar=FORZAR))
    except Exception as e:
        errores_ocr.append({"id": p.stem, "error": f"{type(e).__name__}: {e}"})
ocr_df = pd.DataFrame(res_ocr)
fecha_ocr = datetime.fromtimestamp(min((ocr.CACHE_PADDLE / f"{i}.json").stat().st_mtime for i in ocr_df["id"]))
ocr_df["n_caracteres"] = ocr_df["texto_ocr"].str.len()
ocr_df = ocr_df.merge(clasif[["id", "tipo"]], on="id")
print(f"Documentos procesados: {len(ocr_df)}/25 · errores: {len(errores_ocr)} · fecha de procesamiento OCR: {fecha_ocr:%Y-%m-%d %H:%M}")
print(f"Tiempo total de OCR: {ocr_df['tiempo_ocr_seg'].sum() / 60:.1f} min · mediana por documento: {ocr_df['tiempo_ocr_seg'].median():.1f} s")
print("Preprocesamiento aplicado:", ocr_df["preprocesamiento"].value_counts().to_dict())
display(ocr_df[["id", "tipo", "ancho_px", "alto_px", "n_lineas", "n_caracteres", "confianza_media", "tiempo_ocr_seg"]]
        .style.hide(axis="index").format({"confianza_media": "{:.3f}", "tiempo_ocr_seg": "{:.1f}"}))

Documentos procesados: 25/25 · errores: 0 · fecha de procesamiento OCR: 2026-10-23 22:34
Tiempo total de OCR: 613.7 min · mediana por documento: 87.1 s
Preprocesamiento aplicado: {'ninguno': 19, 'canal alfa opaco descartado -> RGB': 6}


id,tipo,ancho_px,alto_px,n_lineas,n_caracteres,confianza_media,tiempo_ocr_seg
doc_001,manuscrito,697,973,46,444,0.597,39.1
doc_002,manuscrito,1132,661,17,232,0.848,23.8
doc_003,manuscrito,791,459,9,113,0.936,338.1
doc_004,foto,896,1137,22,520,0.779,19.6
doc_005,impreso,1087,1643,11,238,0.972,23.9
doc_006,impreso,1099,1560,56,2009,0.991,54.1
doc_007,impreso,1550,2336,80,2329,0.978,87.1
doc_008,foto,5223,3482,109,1276,0.721,137.3
doc_009,manuscrito,1644,2144,47,1130,0.595,64.6
doc_010,impreso,1633,2161,9,423,0.942,40.5


**Observación: tiempos de OCR.** El OCR de los 25 documentos tomó **613,7 min (10,2 h)** en CPU; la mediana es de 87 s por documento. Dos valores atípicos dominan el total:
- `doc_025` (mapa de 1871, 7962 × 11 698 px, 1376 líneas) tardó **34 161 s (≈ 9,5 h)**.
- `doc_003` (un recibo de 791 × 459 px con 9 líneas) tardó **338 s**.

Sin `doc_025`, el tiempo prácticamente no se relaciona con el número de líneas (r = 0,07). La hipótesis más probable, **no verificada**, es la configuración forzada sin oneDNN (incidencia 4): PaddlePaddle 3.3.1 corre en CPU con un *backend* no optimizado cuyo costo varía según la forma de cada imagen y de cada recorte de línea. Es una **limitación técnica de este entorno** y no una propiedad general de PaddleOCR; los tiempos comparativos de la Parte 2 se medirán de forma controlada.

In [8]:
# Muestra del texto OCR bruto (primeras líneas) por tipo, sin seudonimizar: solo documentos históricos
# de dominio público; el texto publicado en datos/documentos.csv está seudonimizado (sección 5).
for d in ["doc_005", "doc_011", "doc_014"]:
    fila = ocr_df.set_index("id").loc[d]
    display(Markdown(f"**{d}** · {fila['tipo']} · confianza media {fila['confianza_media']:.2f}\n\n```text\n"
                     + "\n".join(fila["texto_ocr"].splitlines()[:8]) + "\n```"))

**doc_005** · impreso · confianza media 0.97

```text
TRADICIONES
PERUANAS
POR
RICARDOPALMA
Miembro Correspondicnte de las Reales Academias Española  de la Historia,
y Director de la Biblioteca Nacional de Lima
TOMO H
BARCELONA
```

**doc_011** · foto · confianza media 0.98

```text
CARTA ESCRITA PORELGENERAL
FELIPESANTIAGOSALAVERRY
HORASANTESDESUMUERTE
AREQUIPA,FEBRERO17-1886
MI QUERIDA
ESPOSA.
DENTRO DE POCOS MOMENTOS VOY A SER
PAGADO POR LAS ARMAS, Y TE DEBO EL ULTIMO
```

**doc_014** · manuscrito · confianza media 0.69

```text
Vn. D Joming Frictan
Vrallyua Cnuarze 2de1822
Mi mefon Amiga inlayra d van
igiheohn a
.qe entenase ed conidert las ventafar qvamor
à perer i paxalicamor mista movimientorns
do un momento g vitor tantor inidentun
Ven za martro Erotecton con nrt delibexa
```

**Observación: calidad visible del OCR.**
- **Impreso nítido (`doc_005`):** casi perfecto, con errores puntuales («RICARDOPALMA», «Correspondicnte»).
- **Placa en relieve fotografiada (`doc_011`):** se leen las palabras, pero se pierden los espacios entre ellas («FELIPESANTIAGOSALAVERRY»).
- **Manuscrito de 1822 (`doc_014`):** el resultado es en gran parte ilegible («Vn. D Joming Frictan»).

La confianza media autoinformada (0,69 en `doc_014`) señala el problema, pero no lo cuantifica: para eso hacen falta la referencia humana y el CER (Parte 2).

## 3. ASR principal con faster-whisper + VAD
**Configuración:**
- Modelo Whisper **small** (`Systran/faster-whisper-small`), en CPU con `int8`.
- `language="es"`, `beam_size=5`, `vad_filter=True` (Silero VAD con parámetros por defecto) y `word_timestamps=True`, que se usa para alinear con la diarización. El resto de parámetros queda por defecto.
- Además se ejecuta una **detección de idioma libre** (sin forzar `es`) como evidencia automática del idioma hablado.

**Métricas de rendimiento:**
- `tiempo_proc_s`: tiempo de transcripción del clip, sin contar la carga del modelo.
- `latencia_primer_segmento_s`: tiempo hasta obtener el primer segmento.
- `RTF = tiempo_proc_s / duración`: un valor menor que 1 indica que procesa más rápido que el tiempo real.

In [9]:
crono.marca("3. ASR faster-whisper (lectura de caché si existe)")
rutas_clips = sorted((RAW / "audio" / "clips").glob("aud_*.wav"))
pend_asr = [p for p in rutas_clips if FORZAR or not (asr.CACHE / f"{p.stem}.json").exists()]
modelo_asr = None
if pend_asr:
    modelo_asr, t_carga_asr = asr.cargar_modelo()
    print(f"faster-whisper cargado en {t_carga_asr:.1f} s; clips por procesar: {len(pend_asr)}")
res_asr, errores_asr = [], []
for p in rutas_clips:
    try:
        res_asr.append(asr.transcribir(modelo_asr, p, forzar=FORZAR))
    except Exception as e:
        errores_asr.append({"id": p.stem, "error": f"{type(e).__name__}: {e}"})
asr_df = pd.DataFrame(res_asr).merge(clips[["id", "fuente_id", "fuente_etiqueta", "inicio_s"]], on="id")
asr_df["n_palabras"] = asr_df["texto_asr"].str.split().str.len()
asr_df["palabras_por_min"] = asr_df["n_palabras"] / (asr_df["duracion_seg"] / 60)
asr_df["pct_voz_vad"] = 100 * asr_df["duracion_tras_vad_seg"] / asr_df["duracion_seg"]
fecha_asr = datetime.fromtimestamp(min((asr.CACHE / f"{i}.json").stat().st_mtime for i in asr_df["id"]))
carga = json.loads((asr.CACHE / "_carga_modelo.json").read_text()) if (asr.CACHE / "_carga_modelo.json").exists() else {}
print(f"Clips transcritos: {len(asr_df)}/{len(rutas_clips)} · errores: {len(errores_asr)} · fecha de procesamiento ASR: {fecha_asr:%Y-%m-%d %H:%M}")
print(f"Carga del modelo: {carga.get('tiempo_carga_s', 'n/d')} s · tiempo total de transcripción: {asr_df['tiempo_proc_s'].sum() / 60:.1f} min "
      f"para {asr_df['duracion_seg'].sum() / 60:.1f} min de audio · RTF global {asr_df['tiempo_proc_s'].sum() / asr_df['duracion_seg'].sum():.3f}")
display(asr_df[["id", "fuente_id", "duracion_seg", "pct_voz_vad", "n_palabras", "idioma_detectado_libre",
                "idioma_detectado_prob", "tiempo_proc_s", "latencia_primer_segmento_s", "rtf"]]
        .style.hide(axis="index").format({"duracion_seg": "{:.1f}", "pct_voz_vad": "{:.0f} %", "idioma_detectado_prob": "{:.3f}",
                                          "tiempo_proc_s": "{:.1f}", "latencia_primer_segmento_s": "{:.1f}", "rtf": "{:.3f}"}))

Clips transcritos: 30/30 · errores: 0 · fecha de procesamiento ASR: 2026-10-24 09:02
Carga del modelo: 1.887 s · tiempo total de transcripción: 16.4 min para 36.7 min de audio · RTF global 0.446


id,fuente_id,duracion_seg,pct_voz_vad,n_palabras,idioma_detectado_libre,idioma_detectado_prob,tiempo_proc_s,latencia_primer_segmento_s,rtf
aud_001,audio_A,67.8,94 %,171,es,0.936,14.2,5.3,0.210
aud_002,audio_A,71.3,92 %,153,es,0.965,27.4,8.5,0.384
aud_003,audio_A,68.4,100 %,141,es,0.948,20.7,6.0,0.303
aud_004,audio_A,74.5,100 %,236,es,0.961,28.2,11.0,0.379
aud_005,audio_A,72.0,99 %,184,es,0.939,22.4,5.9,0.311
aud_006,audio_A,70.5,100 %,174,es,0.969,21.2,5.1,0.301
aud_007,audio_A,66.2,100 %,189,es,0.948,19.3,5.4,0.292
aud_008,audio_B,70.1,99 %,164,es,0.961,16.7,5.2,0.239
aud_009,audio_B,64.9,99 %,136,es,0.971,16.4,4.9,0.252
aud_010,audio_B,65.5,100 %,147,es,0.978,16.4,5.1,0.251


**Observación: ASR.**
- **Velocidad:** transcribir 36,7 min de audio tomó 16,4 min. El RTF global es de 0,446 y la mediana por clip de **0,26**, es decir, unas 4 veces más rápido que el tiempo real en CPU.
- **Latencia:** la mediana hasta el primer segmento es de 5,1 s (máximo 11,0 s en `aud_004`).
- **Valor atípico:** `aud_027` (fuente F) tuvo un **RTF de 5,13**. Su salida final no muestra repeticiones (ningún segmento tiene una razón de compresión mayor que 2). La explicación más probable, **no verificada**, son las re-decodificaciones con mayor temperatura que faster-whisper aplica cuando un tramo no supera sus umbrales de calidad (aplausos y voces del público).
- **VAD:** conservó como voz entre el 84 % (`aud_019`, ceremonia con música) y el 100 % de cada clip, así que no hay clips sin voz.

### Idioma de los audios
El ASR se ejecuta con `language="es"`. La columna `idioma_detectado_libre` muestra lo que Whisper detecta **sin forzar el idioma**; es evidencia automática, no una confirmación. Las fuentes **F y G** se incorporaron en esta tarea y su idioma final queda como `PENDIENTE_REVISION_HUMANA` hasta que una persona escuche los audios.

In [10]:
idioma_fuente = (asr_df.groupby("fuente_id")
                 .agg(clips=("id", "count"),
                      detectado_es=("idioma_detectado_libre", lambda s: f"{(s == 'es').sum()}/{len(s)}"),
                      prob_media=("idioma_detectado_prob", "mean"),
                      otros_idiomas=("idioma_detectado_libre", lambda s: ", ".join(sorted(set(s) - {"es"})) or "—"))
                 .reset_index())
idioma_fuente["idioma_final"] = np.where(idioma_fuente["fuente_id"].isin(["audio_F", "audio_G"]), PENDIENTE, "es")
display(idioma_fuente.style.hide(axis="index").format({"prob_media": "{:.3f}"}))

fuente_id,clips,detectado_es,prob_media,otros_idiomas,idioma_final
audio_A,7,7/7,0.952,—,es
audio_B,3,3/3,0.970,—,es
audio_C,2,2/2,0.958,—,es
audio_D,4,4/4,0.953,—,es
audio_E,4,4/4,0.939,—,es
audio_F,7,7/7,0.918,—,PENDIENTE_REVISION_HUMANA
audio_G,3,3/3,0.953,—,PENDIENTE_REVISION_HUMANA


**Observación: idioma.** La detección libre identificó **español en los 30 clips**, con probabilidades entre 0,842 y 0,989 (mediana 0,958); esto incluye los 10 clips de F y G. Es evidencia automática coherente con los metadatos de Commons. La confirmación final de F y G sigue como `PENDIENTE_REVISION_HUMANA` hasta escuchar los audios.

## 4. Diarización de hablantes
**Método:** sherpa-onnx 2.0, que combina la segmentación de pyannote 3.0 (ONNX, MIT) con *embeddings* WeSpeaker ResNet34-LM (Apache-2.0) y *clustering* con **número de hablantes automático** (`num_clusters=-1`, umbral 0,5). No se fija de antemano que haya 2 hablantes.
- Se diariza **cada fuente completa**, de modo que `[HABLANTE_k]` designa a la misma voz en todos los clips de esa fuente.
- Cada palabra del ASR recibe el hablante con el que más se solapa en el tiempo.
- Se eligió sherpa-onnx en lugar de pyannote.audio porque los modelos de pyannote en Hugging Face son de acceso restringido (requieren token) y porque torch y paddle en el mismo entorno causan conflictos en Windows.

El número de hablantes obtenido es una **estimación automática**. `n_hablantes` queda como `PENDIENTE_REVISION_HUMANA` hasta que una persona escuche los audios.

In [11]:
crono.marca("4a. Diarización (lectura de caché si existe) y alineación con ASR")
diarizador = None
diar = {}
for ruta in sorted((RAW / "audio" / "normalizados").glob("*.flac")):
    if FORZAR or not (diarizacion.CACHE / f"{ruta.stem}.json").exists():
        diarizador = diarizador or diarizacion.crear_diarizador()
    diar[ruta.stem] = diarizacion.diarizar_fuente(diarizador, ruta, forzar=FORZAR)

diar_df = pd.DataFrame([{"fuente_id": k, "fuente": ETIQUETA_FUENTE[k], "duracion_min": v["duracion_seg"] / 60,
                         "n_hablantes_estimado": v["n_hablantes_estimado"], "segmentos": len(v["segmentos"]),
                         "reparto_seg": ", ".join(f"{h.split('_')[1]}: {s:.0f}s" for h, s in v["segundos_por_hablante"].items()),
                         "rtf": v["rtf"]} for k, v in diar.items()])
display(diar_df.style.hide(axis="index").format({"duracion_min": "{:.1f}", "rtf": "{:.3f}"}))

textos_diar, presentes = {}, {}
for _, fila in asr_df.iterrows():
    res = next(r for r in res_asr if r["id"] == fila["id"])
    textos_diar[fila["id"]], presentes[fila["id"]] = diarizacion.transcripcion_diarizada(res, diar[fila["fuente_id"]], fila["inicio_s"])
asr_df["texto_asr_diarizado"] = asr_df["id"].map(textos_diar)
asr_df["hablantes_en_clip"] = asr_df["id"].map(lambda i: ", ".join(presentes[i]))
asr_df["n_hablantes_estimado"] = asr_df["id"].map(lambda i: len(presentes[i]))
por_clip = (asr_df.assign(clip=lambda d: d["id"].str[-3:] + " (" + d["n_hablantes_estimado"].astype(str) + ")")
            .groupby("fuente_id")["clip"].agg(" · ".join).rename("clips (hablantes estimados en el clip)").reset_index())
display(por_clip.style.hide(axis="index"))

fuente_id,fuente,duracion_min,n_hablantes_estimado,segmentos,reparto_seg,rtf
audio_A,A · Entrevista en Jr. de la Unión,8.2,3,130,"1: 363s, 2: 15s, 3: 2s",0.114
audio_B,B · Entrevista en Plaza San Martín (1),3.3,2,44,"1: 163s, 2: 3s",0.166
audio_C,C · Entrevista en Plaza San Martín (2),2.8,2,7,"1: 169s, 2: 2s",0.206
audio_D,D · Excombatiente del Cenepa,5.5,2,22,"1: 304s, 2: 10s",0.196
audio_E,E · Entrevistas en ceremonia patronal,5.4,3,77,"1: 89s, 2: 5s, 3: 177s",0.174
audio_F,F · Conferencia de prensa (Plaza San Martín),8.3,5,44,"1: 443s, 2: 26s, 3: 0s, 4: 4s, 5: 1s",0.231
audio_G,G · Vendedoras en El Agustino,3.2,2,26,"1: 156s, 2: 11s",0.221


fuente_id,clips (hablantes estimados en el clip)
audio_A,001 (2) · 002 (2) · 003 (1) · 004 (2) · 005 (1) · 006 (3) · 007 (1)
audio_B,008 (2) · 009 (1) · 010 (1)
audio_C,011 (1) · 012 (1)
audio_D,013 (1) · 014 (1) · 015 (1) · 016 (2)
audio_E,017 (3) · 018 (2) · 019 (2) · 020 (2)
audio_F,021 (2) · 022 (2) · 023 (2) · 024 (2) · 025 (1) · 026 (1) · 027 (2)
audio_G,028 (2) · 029 (2) · 030 (2)


**Observación: diarización.**
- **Hablantes estimados por fuente:** A 3, B 2, C 2, D 2, E 3, F 5 y G 2. Por clip: 12 clips con 1 hablante, 16 con 2 y 2 con 3.
- **Reparto del habla:** en A–D, el segundo hablante, probablemente el entrevistador que hace preguntas breves, ocupa solo entre el 1 % y el 4 % del tiempo de habla. Esto es coherente con entrevistas de respuestas largas, pero también podría indicar que el sistema mezcla voces.
- **Excepción, E (ceremonia con varios entrevistados):** el reparto es más equilibrado (33 % y 65 %).

In [12]:
crono.marca("4b. Gráficos de diarización")
# Caso multihablante para revisión humana: fuente con más hablantes estimados
# (desempate: mayor proporción de voz del segundo hablante más frecuente).
def peso_segundo(v):
    s = sorted(v["segundos_por_hablante"].values(), reverse=True)
    return s[1] / sum(s) if len(s) > 1 else 0
caso = max(diar, key=lambda k: (diar[k]["n_hablantes_estimado"], peso_segundo(diar[k])))
print(f"Caso multihablante seleccionado por la regla: {ETIQUETA_FUENTE[caso]} · {diar[caso]['n_hablantes_estimado']} hablantes estimados")
print(f"Caso de control (metadatos de Commons: dos entrevistadas + entrevistador): {ETIQUETA_FUENTE['audio_G']} · "
      f"{diar['audio_G']['n_hablantes_estimado']} hablantes estimados")

def linea_tiempo(fuente):
  seg = pd.DataFrame(diar[fuente]["segmentos"])
  seg["duracion_s"] = seg["fin"] - seg["inicio"]
  hablantes = sorted(seg["hablante"].unique(), key=lambda h: int(h.split("_")[1]))
  fig = go.Figure()
  for h, color in zip(hablantes, CATEGORICA):
      s = seg[seg["hablante"] == h]
      fig.add_trace(go.Bar(y=[h] * len(s), x=s["duracion_s"], base=s["inicio"], orientation="h", name=h, width=0.6,
                           marker=dict(color=color, line=dict(color=SUPERFICIE, width=1)),
                           customdata=np.c_[s["inicio"], s["fin"]],
                           hovertemplate="%{y}<br>%{customdata[0]:.1f}–%{customdata[1]:.1f} s<extra></extra>"))
  for _, c in clips[clips["fuente_id"] == fuente].iterrows():
      fig.add_vline(x=c["inicio_s"], line=dict(color=EJE, width=1))
      fig.add_annotation(x=c["inicio_s"], y=1.04, yref="paper", text=c["id"], showarrow=False, xanchor="left",
                         font=dict(size=11, color=TENUE))
  reparto = diar[fuente]["segundos_por_hablante"]
  dominante = max(reparto, key=reparto.get)
  fig.update_layout(title=dict(text=f"{ETIQUETA_FUENTE[fuente]}: {len(hablantes)} hablantes estimados; "
                                    f"{dominante} concentra el {100 * reparto[dominante] / sum(reparto.values()):.0f} % del habla",
                               subtitle=dict(text="Cada barra es un turno detectado automáticamente; las líneas verticales marcan el inicio de cada clip")),
                    xaxis_title="Segundos desde el inicio de la fuente", barmode="overlay", height=150 + 45 * len(hablantes),
                    showlegend=False, yaxis=dict(autorange="reversed", title=None))
  FIGURAS[f"diarizacion_{fuente}"] = fig
  fig.show()

for fuente in [caso, "audio_G"]:
    linea_tiempo(fuente)

Caso multihablante seleccionado por la regla: F · Conferencia de prensa (Plaza San Martín) · 5 hablantes estimados
Caso de control (metadatos de Commons: dos entrevistadas + entrevistador): G · Vendedoras en El Agustino · 2 hablantes estimados


Las transcripciones diarizadas de estos casos se muestran en la **sección 5**, ya seudonimizadas, porque este notebook se publica.

**Lectura de los dos casos (requiere escucha humana).**
- **F (elegido por la regla, 5 hablantes estimados):** HABLANTE_1 concentra el 93 % del habla. HABLANTE_3 y HABLANTE_5 suman menos de 1 s cada uno, así que probablemente son **agrupamientos espurios** (aplausos o voces del público) y no personas distintas.
- **G (caso de control):** la diarización estima 2 hablantes y asigna el 94 % al HABLANTE_1. Sin embargo, la descripción de Commons y la propia transcripción indican **al menos tres voces** (el entrevistador y dos vendedoras). Es probable una **subestimación**: voces de timbre parecido, grabadas a corta distancia con el mismo micrófono.
- **Qué escuchar:** los clips aud_021–aud_027 (F) en los instantes marcados como HABLANTE_2 a HABLANTE_5, y los clips aud_028–aud_030 (G) completos. Después, registrar el número real en `n_hablantes`.

## 5. Seudonimización
**Procedimiento reproducible** (`scripts/privacidad.py`):
- **Expresiones regulares** para `[EMAIL]`, `[DNI]` (8 dígitos solo si van precedidos de «DNI» o «documento de identidad»), `[TELEFONO]` (móviles 9XX XXX XXX y fijos con prefijo) y `[DIRECCION]` (vía + nombre + número de casa).
- **NER de spaCy** (`es_core_news_md`) para nombres de persona, reemplazados por `[NOMBRE_n]` con el **mismo marcador en todo el corpus**.
- Se **exceptúan las personas públicas o históricas** cuyo nombre forma parte del valor documental (Ricardo Palma, Miguel Grau, Agustín Gamarra, etc.). Esa lista es una propuesta que queda `PENDIENTE_REVISION_HUMANA`.
- Los números de 8 dígitos sin contexto **no se reemplazan**: se registran como candidatos para revisión.
- Se ignoran los falsos positivos frecuentes del NER observados en este corpus («Entonces», «Gracias», lugares como «San Marcos»).

**Política por época.** La Ley 29733 de Protección de Datos Personales protege a personas naturales **vivas**.
- **Documentos históricos** (anteriores a 1950; personas fallecidas o figuras públicas): solo se aplican `[EMAIL]`, `[TELEFONO]` y `[DNI]`. Seudonimizar a Gamarra o a Grau destruiría el valor documental sin proteger a nadie.
- **Documentos contemporáneos y todos los audios**: se aplica el procedimiento completo.
- La asignación de época de cada documento es una decisión propuesta que queda `PENDIENTE_REVISION_HUMANA`.

**Qué se publica:** solo el texto seudonimizado y un registro **sin los valores originales**. El mapa entre original y marcador se guarda en `datos/raw/privacidad/`, que no se versiona. La revisión final es **humana**, porque el NER falla con nombres en minúscula, con OCR ruidoso y con nombres poco frecuentes.

In [13]:
crono.marca("5. Seudonimización (dos pasadas + verificación de fugas)")
CONTEMPORANEOS = {
    "doc_013": "carta privada fechada en 2014 (personas posiblemente vivas)",
    "doc_021": "fotografía reciente de una pantalla; muestra un correo electrónico",
    "doc_022": "acta municipal de 2015–2016 con nombres de particulares",
    "doc_023": "acta municipal de 2015–2016 con nombres de particulares",
}
epoca = pd.DataFrame({"id": ocr_df["id"]})
epoca["epoca"] = np.where(epoca["id"].isin(list(CONTEMPORANEOS)), "contemporáneo", "histórico")
epoca["justificacion"] = epoca["id"].map(CONTEMPORANEOS).fillna("contenido anterior a 1950 (personas fallecidas o figuras públicas)")
epoca["politica"] = np.where(epoca["epoca"] == "histórico", "solo EMAIL/TELEFONO/DNI", "completa (NER + regex)")
epoca["revision"] = PENDIENTE
display(epoca.groupby(["epoca", "politica"])["id"].agg(lambda s: ", ".join(s)).rename("documentos").reset_index().style.hide(axis="index"))
es_historico = dict(zip(epoca["id"], epoca["epoca"] == "histórico"))

seud = privacidad.Seudonimizador()
registro = []
def seudonimizar(id_, modalidad, campo, texto, historico=False):
    salida, ents = seud.procesar(texto, historico=historico)
    registro.extend({"id": id_, "modalidad": modalidad, "campo": campo, **e} for e in ents)
    return salida, sum(e["accion"] == "reemplazado" for e in ents)

# Semillas: nombres de persona presentes en títulos/descripciones de Commons (extraídos con el mismo NER;
# no se imprimen ni se escriben en el código). Pasada 1: detectar nombres en todo el corpus.
semillas = privacidad.nombres_de_metadatos(seud.nlp, RAW / "metadatos" / "commons_audio_metadata.json")
seud.agregar_conocidos(semillas, semilla=True)
for i, t in zip(ocr_df["id"], ocr_df["texto_ocr"]):
    seud.procesar(t, historico=es_historico[i])
for t in asr_df["texto_asr"]:
    seud.procesar(t)
seud.agregar_conocidos(list(seud.mapa_nombres))  # pasada 2: reemplazar TODAS las apariciones de cada nombre
print(f"Semillas desde metadatos: {len(semillas)} · nombres conocidos para propagación: {len(seud.conocidos)}")

salidas = [seudonimizar(i, "documento", "texto_ocr", t, es_historico[i]) for i, t in zip(ocr_df["id"], ocr_df["texto_ocr"])]
ocr_df["texto_ocr_seud"], ocr_df["n_reemplazos"] = [s for s, _ in salidas], [n for _, n in salidas]
salidas = [seudonimizar(i, "audio", "texto_asr", t) for i, t in zip(asr_df["id"], asr_df["texto_asr"])]
asr_df["texto_asr_seud"], asr_df["n_reemplazos"] = [s for s, _ in salidas], [n for _, n in salidas]
asr_df["texto_asr_diarizado_seud"] = [seud.procesar(t)[0] for t in asr_df["texto_asr_diarizado"]]

reg = pd.DataFrame(registro)
(RAW / "privacidad").mkdir(parents=True, exist_ok=True)
reg.to_csv(RAW / "privacidad" / "mapa_entidades_PRIVADO.csv", index=False, encoding="utf-8")       # no versionado
pd.DataFrame([{"original": k, "marcador": v} for k, v in seud.mapa_nombres.items()]).to_csv(
    RAW / "privacidad" / "mapa_nombres_PRIVADO.csv", index=False, encoding="utf-8")                  # no versionado
publico = reg.drop(columns=["original"]) if len(reg) else reg
publico["revision"] = REVISION_PRIVACIDAD
publico.to_csv(REPORTES / "seudonimizacion_registro.csv", index=False, encoding="utf-8")
resumen_priv = (reg.groupby(["modalidad", "tipo", "accion"]).size().rename("n").reset_index()
                if len(reg) else pd.DataFrame(columns=["modalidad", "tipo", "accion", "n"]))
display(resumen_priv.style.hide(axis="index"))
print(f"Nombres distintos seudonimizados: {len(seud.mapa_nombres)} · documentos con reemplazos: {(ocr_df['n_reemplazos'] > 0).sum()}/25 "
      f"· clips con reemplazos: {(asr_df['n_reemplazos'] > 0).sum()}/{len(asr_df)}")

# Verificación automática de fugas en los textos que deben ir seudonimizados (contemporáneos + audio):
# ningún nombre conocido (semilla o detectado) puede seguir presente. Solo se imprime el conteo.
a_publicar = "\n".join(list(ocr_df.loc[~ocr_df["id"].map(es_historico), "texto_ocr_seud"])
                       + list(asr_df["texto_asr_seud"]) + list(asr_df["texto_asr_diarizado_seud"]))
fugas = sum(bool(pats[0].search(a_publicar)) for pats in seud.conocidos.values())
print(f"Verificación de fugas: {fugas} nombres conocidos presentes en el texto a publicar")
assert fugas == 0, "Hay nombres conocidos sin seudonimizar: no publicar"

epoca,politica,documentos
contemporáneo,completa (NER + regex),"doc_013, doc_021, doc_022, doc_023"
histórico,solo EMAIL/TELEFONO/DNI,"doc_001, doc_002, doc_003, doc_004, doc_005, doc_006, doc_007, doc_008, doc_009, doc_010, doc_011, doc_012, doc_014, doc_015, doc_016, doc_017, doc_018, doc_019, doc_020, doc_024, doc_025"


Semillas desde metadatos: 8 · nombres conocidos para propagación: 29


modalidad,tipo,accion,n
audio,NOMBRE,reemplazado,27
documento,EMAIL,reemplazado,1
documento,NOMBRE,reemplazado,24


Nombres distintos seudonimizados: 42 · documentos con reemplazos: 4/25 · clips con reemplazos: 14/30
Verificación de fugas: 0 nombres conocidos presentes en el texto a publicar


### Transcripciones diarizadas de los casos multihablante (versión seudonimizada)
Estos son los fragmentos que la autora debe **escuchar y revisar**: si las etiquetas `[HABLANTE_k]` corresponden a voces reales distintas, si faltan o sobran hablantes y si quedó algún nombre sin seudonimizar.

In [14]:
tabla_seud = asr_df.set_index("id")
for fuente in [caso, "audio_G"]:
    for i in clips.loc[clips["fuente_id"] == fuente, "id"]:
        t = tabla_seud.loc[i, "texto_asr_diarizado_seud"]
        display(Markdown(f"**{i}** · {ETIQUETA_FUENTE[fuente]} · hablantes en el clip: {tabla_seud.loc[i, 'hablantes_en_clip']}\n\n```text\n"
                         + t[:900] + ("\n[…]" if len(t) > 900 else "") + "\n```"))

**aud_021** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_1] ¿Qué esperamos? Bien,
[HABLANTE_2] estamos en Plata San Martín. Antes que nada, ¿me podría decir su nombre, por favor?
[HABLANTE_1] Mi nombre es la señora [NOMBRE_33]. Vengo desde el departamento de Apurima en Marcha en solidarizarnos con nuestro gobierno y pedimos que cierren el Congreso contra esos congresistas que no trabajan para su pueblo. Queremos que se retiren ya del Congreso y queremos que deje a nuestro gobierno gobernar. Es
[HABLANTE_2] una pregunta. Para usted, ¿qué significa el Perú?
[HABLANTE_1] El Perú es una patria grande donde todos los peruanos acá somos parte de eso y nuestro Perú no debe ser así. Estropeado, pisado por gente que no sirven a su pueblo. Es un país grande, rico en minería, en alimento. En todo somos rico y por lo tanto somos parte del Perú. Y lo amamos y lo queremos y por eso estamos hoy en marcha de todas nuestras provincias aquí en Abanca,
[…]
```

**aud_022** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_2] Y una pregunta, ¿Usted en qué momento sintió por primera vez, en qué momento descubrió que usted era peruana?
[HABLANTE_1] Yo, desde muy pequeña, mi padre ha sido un gran político, siempre nos ha hablado bastante de nuestro período, de sentirnos, de amarnos y creer, incluso él decía, hay que estudiar. ¿Para qué? Para servir a nuestra patria. Mi padre era un hombre que le gustaba compartir, era peruano porque se sentía peruano y todas las riquezas eran para todos. No ha sido un hombre egoísta, ha
[HABLANTE_?] sido
[HABLANTE_1] un hombre de bien o cuando se ha inculcado, desde muy niños, amar y a querer y a compartir, gracias.
[HABLANTE_2] ¿Qué aspectos positivos diría que tiene el Perú? ¿Cómo? ¿Qué aspectos positivos diría que tiene el Perú?
[HABLANTE_1] El Perú tiene muchos aspectos positivos, es otras cosas que a veces nuestras autoridades no lo ven por ese lado. Pues es po
[…]
```

**aud_023** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_1] Hay posibilidades positivas,
[HABLANTE_?] pero
[HABLANTE_1] a veces las autoridades lo ven negativo, lo ven la parte de lucrarse cada uno.
[HABLANTE_2] ¿Y ahora qué características negativas diría Ferín el Verú?
[HABLANTE_1] Tenemos, a lo menos, características respecto a nuestros gobernantes que son egoístas, solo todo jalan para ellos, se lucran para ellos, no trabajan para el pueblo, eso sería un aspecto negativo. ¿Y por
[HABLANTE_2] último qué mensaje usted le daría a las futuras generaciones de peruanos?
[HABLANTE_1] Pucha, yo les diría a los jóvenes como usted, a los peruanos como usted, ahora ya no es como antes que teníamos una radio, pero ahora ustedes tienen esa potencia, esta tecnología donde vemos, ahora se ha descubierto toda la verdad jóvenes, espero porque el futuro es de ustedes, nosotros ya nos vamos y tienen que defender a vuestra patria, luchar, esos vuest
[…]
```

**aud_024** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_1] gracias a la tecnología, la verdad, saben lo que está pasando. Jóvenes, la patria es de ustedes y a continuar luchando para un país grande, lleno de riquezas, de minería. Tenemos todo lo grande que ustedes pueden disfrutar, que de repente podamos ser algún día como ese país Dubai, lleno de minería, de hora. Eso tienen que ustedes luchar jóvenes. Gracias. Gracias,
[HABLANTE_2] señora [NOMBRE_34], créame que... No
[HABLANTE_1] sé si podría pronunciar este pronunciamiento. Claro,
[HABLANTE_2] claro,
[HABLANTE_1] no
[HABLANTE_2] hay
[HABLANTE_1] problema. Nosotros tenemos un pronunciamiento desde la ciudad de Apulimaj, somos una organización de mujeres desocupadas, micaela bastidas, a cual hemos traído acá para poder pronunciar. Mira, nuestro país enfrenta el regreso de la anocromismo salvaje y brutal, donde las mujeres éramos un objeto más de práctica, serviles de quienes se au
[…]
```

**aud_025** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1

```text
[HABLANTE_1] [NOMBRE_42] nuestros derechos fundamentalmente y principalmente de nosotras, las mujeres para ellos se han unido desde la derecha a patria, a patria acá y su puesta izquierda que sorprendentemente se autodenominan provincianas. Ambos han inflado sus baterías para intentar destruir el ministerio de la mujer y poblaciones vulnerables y una entidad producto de lucha de todas las mujeres peruanas desde la región apurima nos levantamos contra estas ignomias que buscan desconocer el rol fundamentalmente de la mujer en la sociedad y en anda de todas las normatibilidades como la ley 30 36 ley para erradicar sancionar la violencia contra las mujeres y los integrantes del grupo familiar ley 28 98 de igualdad
```

**aud_026** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1

```text
[HABLANTE_1] de oportunidades entre mujeres y hombres, principalmente la violan, la normatibilidad internacional, que nos protege como la convención internacional para prevenir y sancionar y erradicar la violencia contra las mujeres, especialmente el convenio de [NOMBRE_35]. En tal sentido, las mujeres peruanas, principalmente las apurimeñas, nada nos han regalado, todo lo hemos conquistado con lucha sacrificio, nos siguiendo el legado de nuestra heroína, Micaela Bastidas Puyucava. Las mujeres peruanas somos 16 millones de ciudadanas y nos dejamos pasar esta frente. Hoy decimos a los descendientes de estas bárbaros que se camuflan en el Congreso que no pasará el intento de cambiar de nombre el ministerio de la mujer, porque en quiénes, en 500 años
```

**aud_027** · F · Conferencia de prensa (Plaza San Martín) · hablantes en el clip: HABLANTE_1, HABLANTE_4

```text
[HABLANTE_1] para construir una patria de igualdad en libertad. Somos peruanos y nos regresaremos al mundo de la caver, cavernícola, de tanto extrañan estos herederos del coloniaje. Impulsaremos y recolostaremos la firma de todas las mujeres peruanas para defender nuestros hijos. ¡Viva el Ministerio de la Mujer! ¡Viva la población de es vulnerables! Muchísimas gracias.
[HABLANTE_4] Gracias, señora [NOMBRE_36], que sus palabras me llenan de espíritus realmente. Gracias papi,
[HABLANTE_1] a continuar joven lindo, ustedes son los dueños, en ustedes está el futuro, ustedes son quienes se van a quedar, nosotros somos, aperturamos la nuestra lucha en favor de ustedes, porque ustedes son nuestros hijos, yo tengo hijos y quiero que mis hijos algún día velos grande y que luchen por esta hermosa patria que tanto amamos y es una patria grande, lleno de millonario, lleno de oro, de riqueza, adelante
[…]
```

**aud_028** · G · Vendedoras en El Agustino · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_1] Buenos días profesor y compañeros. Hoy estoy cerca de aquí por mi casa y voy a hacer entrevistas cerca del fútbol como unión de masas. Estoy con las señoras [NOMBRE_37]. ¿Estáis sa[NOMBRE_38]? Muy bien, voy a comenzar con la primera pregunta y es ¿qué expectativas tienen ante este partido? Que la afición se encuentra motivado para ganar este partido. ¿Me puede decir su nombre? [NOMBRE_39]. Esa expectativa que pienso es que acá que el Perú va a dar todo por todo, todo su corazón va a entregar para que ganemos ¿no? para que seamos ganemos este para ir a las olimpiadas al
[HABLANTE_2] mundial. ¿Me puede decir este su nombre?
[HABLANTE_1] [NOMBRE_39] Rientos. Bien, sigo con la siguiente pregunta. ¿Qué jugador piensa que va a
[HABLANTE_?] hacer?
[HABLANTE_1] Antes de que diga su respuesta dígame su nombre por favor. [NOMBRE_39].
```

**aud_029** · G · Vendedoras en El Agustino · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_1] La padula.
[HABLANTE_2] ¿Por qué?
[HABLANTE_1] Porque ha demostrado su fuerza y su barra para jugar y ha metido unos cuantos goles. Ha metido goles.
[HABLANTE_2] Muy bien, sigo con la siguiente señora.
[HABLANTE_1] [NOMBRE_38]. Para mí sería la padula con cueva, porque los dos han hecho un duo. Un duo y la padula más para que demuestre que tiene el corazón más peruano que italiano. Muchas gracias, sigo con la siguiente, sigo con la pregunta. Y antes de empezar con ella puedo decir que han destacado la padula más, ¿no? Y la siguiente pregunta es ¿cuál es el resultado de hoy? ¿Cuál piensas que va a ser el resultado de hoy? [NOMBRE_39] 2-0.
[HABLANTE_2] Sigo con la siguiente señorita.
[HABLANTE_1] [NOMBRE_38].
```

**aud_030** · G · Vendedoras en El Agustino · hablantes en el clip: HABLANTE_1, HABLANTE_2

```text
[HABLANTE_1] También 2-0, para lo que sea 2-0. Veo que ambas han dicho el mismo resultado y esperemos que sea así. Y la última pregunta, ¿qué opina del fútbol acerca como factor de unión de masas? Sabé el basket, el fútbol es un deporte que apasiona a todos los peruanos tanto como fuera del país también y es un deporte que une a toda la familia,
[HABLANTE_?] a
[HABLANTE_1] todas las amistades en general. Muchas gracias y ya finalizando su comentario final. El sabarriento. Sé que la gente se una, no hay distinción de raza ni religión, por que todos sean unidos y que el deporte siga adelante.
[HABLANTE_2] Gracias.
[HABLANTE_1] Muchas gracias, con esto termina la entrevista.
```

**Observación: seudonimización.** La auditoría de la primera versión encontró **fugas en los dos sentidos**:
- **Nombres que quedaban sin reemplazar.** El NER no detecta todas las apariciones de un mismo nombre (nombres en mayúsculas o precedidos de «señorita»).
- **Palabras comunes y topónimos reemplazados por error.** Por ejemplo, «Mache» se convertía en `[NOMBRE_n]` en todo el acta.

Se corrigió con reglas documentadas (tratamiento + nombre, semillas extraídas de los metadatos, propagación con al menos dos palabras) y con una **verificación automática de fugas, que ahora da 0**.

Aun así, el ASR deforma nombres (en un clip el apellido aparece pegado a la palabra anterior) y ningún método automático garantiza una anonimización completa. Por eso la revisión humana es obligatoria antes de publicar.

## 6. Control de duplicados y descartes
Se revisan los **duplicados exactos** (SHA-256), los **casi duplicados** visuales entre documentos (*dHash* de 64 bits, que marca la pareja si la distancia de Hamming es ≤ 10), los archivos **vacíos o ilegibles** y el **audio sin voz** (porcentaje de voz según el VAD menor que 20 %). Ningún elemento se elimina en silencio: todo descarte queda registrado con motivo, fecha y decisión.

In [15]:
crono.marca("6. Duplicados y descartes (dHash + VAD)")
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
def dhash(ruta, n=8):
    with Image.open(ruta) as im:
        im.draft("L", (n * 32, n * 32))
        g = np.asarray(im.convert("L").resize((n + 1, n), Image.Resampling.LANCZOS), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()
hashes = {p.stem: dhash(p) for p in documentos}
ids = list(hashes)
pares = sorted(((int((hashes[a] != hashes[b]).sum()), a, b) for i, a in enumerate(ids) for b in ids[i + 1:]))
casi = [(a, b, d) for d, a, b in pares if d <= 10]
print("Pares de documentos más parecidos (distancia de Hamming dHash, 0 = idénticos):")
display(pd.DataFrame([(a, b, d) for d, a, b in pares[:5]], columns=["doc_a", "doc_b", "distancia"]).style.hide(axis="index"))
print("Casi duplicados (≤ 10):", casi if casi else "ninguno")

sin_voz = asr_df.loc[asr_df["pct_voz_vad"] < 20, "id"].tolist()
print("Clips con < 20 % de voz según VAD:", sin_voz if sin_voz else "ninguno")
print("Errores de OCR:", errores_ocr if errores_ocr else "ninguno", "· errores de ASR:", errores_asr if errores_asr else "ninguno")

desc_audio = pd.read_csv(REPORTES / "descartes_audio.csv")
descartes = pd.concat([
    desc_audio[["id", "modalidad", "motivo", "fecha", "decision"]],
    pd.DataFrame([{"id": i, "modalidad": "audio", "motivo": "menos de 20 % de voz según VAD", "fecha": date.today().isoformat(),
                   "decision": f"conservado; {PENDIENTE}"} for i in sin_voz], columns=["id", "modalidad", "motivo", "fecha", "decision"]),
    pd.DataFrame([{"id": f"{a}~{b}", "modalidad": "documento", "motivo": f"casi duplicado visual (dHash {d})",
                   "fecha": date.today().isoformat(), "decision": f"conservado; {PENDIENTE}"} for a, b, d in casi],
                 columns=["id", "modalidad", "motivo", "fecha", "decision"]),
], ignore_index=True)
descartes.to_csv(REPORTES / "descartes.csv", index=False, encoding="utf-8")
print(f"Registro de descartes: {len(descartes)} filas → datos/reportes/descartes.csv")
display(descartes.groupby(["modalidad", "decision"]).size().rename("n").reset_index().style.hide(axis="index"))

Pares de documentos más parecidos (distancia de Hamming dHash, 0 = idénticos):


doc_a,doc_b,distancia
doc_022,doc_023,12
doc_008,doc_011,13
doc_014,doc_016,14
doc_004,doc_025,15
doc_015,doc_016,15


Casi duplicados (≤ 10): ninguno
Clips con < 20 % de voz según VAD: ninguno
Errores de OCR: ninguno · errores de ASR: ninguno
Registro de descartes: 20 filas → datos/reportes/descartes.csv


modalidad,decision,n
audio,"descartado; reemplazado por clips regenerados desde audio normalizado; se conserva fuera del repositorio, no se versiona",20


**Observación: duplicados y descartes.**
- No hay duplicados exactos ni casi duplicados visuales entre documentos: ninguna pareja tiene una distancia dHash de 10 o menos.
- Ningún clip tiene menos del 20 % de voz.
- El registro de descartes contiene los 20 clips iniciales (19 vacíos y 1 sustituido por la nueva segmentación), cada uno con motivo, fecha y decisión.

## 7. Exportación de tablas finales
- `datos/documentos.csv`: una fila por documento, con el OCR **seudonimizado**. `texto_referencia` queda como `PENDIENTE_REVISION_HUMANA` (la referencia manual se hace en la Parte 2).
- `datos/transcripciones.csv`: una fila por clip, con el ASR **seudonimizado** (versión plana y diarizada). `n_hablantes` queda como `PENDIENTE_REVISION_HUMANA` y `n_hablantes_estimado` es la salida automática.
- `datos/fuentes.csv`: una fila por archivo fuente (25 documentos y 7 audios), con URL, licencia, autor y hashes.

Los textos brutos sin seudonimizar permanecen solo en la caché privada (`datos/raw/cache/`), para la evaluación local.

In [16]:
crono.marca("7. Escritura de CSV finales")
fd = f_docs.set_index("archivo")
epoca.to_csv(REPORTES / "privacidad_epoca_documentos.csv", index=False, encoding="utf-8")
docs_out = (ocr_df.merge(clasif[["id", "observacion_tipo", "es_mixto"]], on="id")
            .merge(epoca[["id", "epoca", "politica"]].rename(columns={"politica": "politica_privacidad"}), on="id")
            .assign(fuente=lambda d: d["archivo"].map(fd["titulo_commons"]),
                    fuente_url=lambda d: d["archivo"].map(fd["fuente_url"]),
                    licencia=lambda d: d["archivo"].map(fd["licencia"]),
                    autor=lambda d: d["archivo"].map(fd["autor"]).fillna(""),
                    formato_real=lambda d: d["archivo"].map(docs_int["formato_real"]),
                    configuracion_ocr=lambda d: d["configuracion"].map(lambda c: json.dumps(c, ensure_ascii=False)),
                    texto_referencia=PENDIENTE, anonimizado=True, revision_privacidad=REVISION_PRIVACIDAD))
# El autor de la fuente (atribución CC) se publica solo en datos/fuentes.csv, no junto al texto seudonimizado
docs_out = docs_out[["id", "archivo", "tipo", "observacion_tipo", "es_mixto", "fuente", "fuente_url", "licencia",
                     "formato_real", "ancho_px", "alto_px", "texto_ocr_seud", "modelo_ocr", "configuracion_ocr",
                     "preprocesamiento", "n_lineas", "n_caracteres", "confianza_media", "tiempo_ocr_seg",
                     "texto_referencia", "epoca", "politica_privacidad", "anonimizado", "n_reemplazos",
                     "revision_privacidad"]].rename(columns={"texto_ocr_seud": "texto_ocr"})
docs_out.to_csv(DATOS / "documentos.csv", index=False, encoding="utf-8")

tr_out = (asr_df.merge(clips[["id", "archivo", "fuente", "licencia"]], on="id")
          .assign(configuracion_asr=lambda d: d["configuracion"].map(lambda c: json.dumps(c, ensure_ascii=False)),
                  idioma=lambda d: np.where(d["fuente_id"].isin(["audio_F", "audio_G"]), PENDIENTE, d["idioma"]),
                  n_hablantes=PENDIENTE, texto_referencia=PENDIENTE, anonimizado=True, revision_privacidad=REVISION_PRIVACIDAD))
tr_out = tr_out[["id", "archivo", "fuente_id", "fuente_etiqueta", "licencia", "inicio_s", "duracion_seg", "idioma",
                 "idioma_detectado_libre", "idioma_detectado_prob", "texto_asr_seud", "texto_asr_diarizado_seud",
                 "modelo_asr", "configuracion_asr", "n_hablantes", "n_hablantes_estimado", "hablantes_en_clip",
                 "duracion_tras_vad_seg", "pct_voz_vad", "n_palabras", "tiempo_proc_s", "latencia_primer_segmento_s",
                 "rtf", "texto_referencia", "anonimizado", "n_reemplazos", "revision_privacidad"]
                ].rename(columns={"archivo": "archivo_audio", "texto_asr_seud": "texto_asr",
                                  "texto_asr_diarizado_seud": "texto_asr_diarizado"})
tr_out.to_csv(DATOS / "transcripciones.csv", index=False, encoding="utf-8")

fa = f_audio.assign(id=f_audio["archivo_local"].str.replace(r"\.\w+$", "", regex=True))
sha_raw = integridad.set_index("archivo")["sha256"].to_dict()
sha_raw.update(pd.read_csv(REPORTES / "audio_adicional_seleccion.csv").set_index("archivo_local")["sha256"].to_dict())
fuentes_out = pd.concat([
    pd.DataFrame({"id": f_docs["archivo"].str.replace(r"\.\w+$", "", regex=True), "modalidad": "documento",
                  "archivo_local": f_docs["archivo"], "titulo": f_docs["titulo_commons"], "fuente_url": f_docs["fuente_url"],
                  "licencia": f_docs["licencia"], "licencia_url": f_docs["licencia_url"].fillna(""), "autor": f_docs["autor"].fillna(""),
                  "sha1": f_docs["sha1_local"]}),
    pd.DataFrame({"id": fa["id"], "modalidad": "audio", "archivo_local": fa["archivo_local"], "titulo": fa["titulo_commons"],
                  "fuente_url": fa["fuente_url"], "licencia": fa["licencia"], "licencia_url": fa["licencia_url"].fillna(""),
                  "autor": fa["autor"], "sha1": fa["sha1_commons"], "fecha_consulta": fa["fecha_consulta"]}),
], ignore_index=True)
fuentes_out["sha256"] = fuentes_out["archivo_local"].map(sha_raw)
fuentes_out["acceso"] = np.where(fuentes_out["licencia"] == PENDIENTE, PENDIENTE,
                                 "abierto · Wikimedia Commons (verificado por SHA-1)")
consulta_docs = datetime.fromtimestamp((REPORTES / "fuentes_documentos_commons.csv").stat().st_mtime).date().isoformat()
fuentes_out["fecha_consulta"] = fuentes_out["fecha_consulta"].fillna(consulta_docs)  # fecha real de la consulta a la API
fuentes_out = fuentes_out[["id", "modalidad", "titulo", "fuente_url", "licencia", "licencia_url", "acceso", "autor",
                           "fecha_consulta", "archivo_local", "sha1", "sha256"]]
fuentes_out.to_csv(DATOS / "fuentes.csv", index=False, encoding="utf-8")

for nombre, df in [("documentos.csv", docs_out), ("transcripciones.csv", tr_out), ("fuentes.csv", fuentes_out)]:
    print(f"datos/{nombre}: {len(df)} filas × {df.shape[1]} columnas")
display(docs_out.head(3)); display(tr_out.head(3)); display(fuentes_out.drop(columns=["titulo", "autor"]).tail(3))

datos/documentos.csv: 25 filas × 25 columnas
datos/transcripciones.csv: 30 filas × 27 columnas
datos/fuentes.csv: 32 filas × 12 columnas


,id,archivo,tipo,observacion_tipo,es_mixto,fuente,fuente_url,licencia,formato_real,ancho_px,...,n_lineas,n_caracteres,confianza_media,tiempo_ocr_seg,texto_referencia,epoca,politica_privacidad,anonimizado,n_reemplazos,revision_privacidad
0,doc_001,doc_001.jpg,manuscrito,Manuscrito antiguo (Acta de Independencia) con numerosas firmas superpuestas.,False,Acta_de_Independencia_Peru.jpg,https://commons.wikimedia.org/wiki/File:Acta_de_Independencia_Peru.jpg,Public domain,JPEG,697,...,46,444,0.5970,39.118,PENDIENTE_REVISION_HUMANA,histórico,solo EMAIL/TELEFONO/DNI,True,0,REVISADO_HUMANAMENTE
1,doc_002,doc_002.jpg,manuscrito,"Mixto: recibo con formulario impreso (Peruvian Telephone Company, 1895) y contenido variable manuscrito (fecha, nomb...",True,Peruvian_Telephone_Company_receipt_Lima_1895.JPG,https://commons.wikimedia.org/wiki/File:Peruvian_Telephone_Company_receipt_Lima_1895.JPG,Public domain,JPEG,1132,...,17,232,0.8481,23.825,PENDIENTE_REVISION_HUMANA,histórico,solo EMAIL/TELEFONO/DNI,True,0,REVISADO_HUMANAMENTE
2,doc_003,doc_003.jpg,manuscrito,Mixto: recibo impreso (Central & South American Telegraph Co.) con campos manuscritos y timbre; se clasifica por el ...,True,Peru_revenue_stamp_on_receipt.jpg,https://commons.wikimedia.org/wiki/File:Peru_revenue_stamp_on_receipt.jpg,Public domain,JPEG,791,...,9,113,0.9360,338.054,PENDIENTE_REVISION_HUMANA,histórico,solo EMAIL/TELEFONO/DNI,True,0,REVISADO_HUMANAMENTE


,id,archivo_audio,fuente_id,fuente_etiqueta,licencia,inicio_s,duracion_seg,idioma,idioma_detectado_libre,idioma_detectado_prob,...,duracion_tras_vad_seg,pct_voz_vad,n_palabras,tiempo_proc_s,latencia_primer_segmento_s,rtf,texto_referencia,anonimizado,n_reemplazos,revision_privacidad
0,aud_001,aud_001.wav,audio_A,A · Entrevista en Jr. de la Unión,CC BY-SA 4.0,0.00,67.79,es,es,0.9356,...,64.030,94.453459,171,14.242,5.287,0.2101,PENDIENTE_REVISION_HUMANA,True,2,REVISADO_HUMANAMENTE
1,aud_002,aud_002.wav,audio_A,A · Entrevista en Jr. de la Unión,CC BY-SA 4.0,67.79,71.32,es,es,0.9654,...,65.800,92.260236,153,27.360,8.456,0.3836,PENDIENTE_REVISION_HUMANA,True,0,REVISADO_HUMANAMENTE
2,aud_003,aud_003.wav,audio_A,A · Entrevista en Jr. de la Unión,CC BY-SA 4.0,139.11,68.39,es,es,0.9481,...,68.086,99.555491,141,20.729,5.984,0.3031,PENDIENTE_REVISION_HUMANA,True,0,REVISADO_HUMANAMENTE


,id,modalidad,fuente_url,licencia,licencia_url,acceso,fecha_consulta,archivo_local,sha1,sha256
29,audio_E,audio,https://commons.wikimedia.org/wiki/File:Entrevistas.webm,CC0,http://creativecommons.org/publicdomain/zero/1.0/deed.en,abierto · Wikimedia Commons (verificado por SHA-1),2026-10-23,audio_E.webm,cdb311a4c725a66c8f2cf0b939dcf3ff97bc42e4,d6e13e2b8e3ecac37b847ca4ed2f1cb7e15188a3ebcbf274b448116d31a836ad
30,audio_F,audio,https://commons.wikimedia.org/w/index.php?curid=121219891,CC BY-SA 4.0,https://creativecommons.org/licenses/by-sa/4.0,abierto · Wikimedia Commons (verificado por SHA-1),2026-10-23,audio_F.ogg,87cda7b15b26b860c0f6712648b4743af2a82310,c47e3d7da8a5bac10808ab6b68a4d06119c7f8dc5484218afd92cca9a9cdde12
31,audio_G,audio,https://commons.wikimedia.org/wiki/File:Entrevista_2_en_la_Av._Rivag%C3%BCero_-_El_Agustino.ogg,CC BY-SA 4.0,https://creativecommons.org/licenses/by-sa/4.0,abierto · Wikimedia Commons (verificado por SHA-1),2026-10-23,audio_G.ogg,532e1f8c24337c6640d606f8ae5f06b2443abc12,3315e1bf37159206a3b8cada69be212c4caa3aef7c6770905387b0b4a2692333


## 8. Análisis exploratorio (EDA)
Los títulos de los gráficos se generan **a partir de los datos calculados**, de modo que expresan lo que efectivamente muestran.

In [17]:
crono.marca("8. Gráficos EDA (interactivos)")
# Gráfico 1 — Distribución de documentos por tipo
g1 = (clasif.groupby("tipo").agg(n=("id", "count"), mixtos=("es_mixto", "sum"), ids=("id", lambda s: ", ".join(s)))
      .reindex(ORDEN_TIPO).reset_index())
g1["pct"] = 100 * g1["n"] / g1["n"].sum()
fig = go.Figure(go.Bar(
    x=g1["n"], y=g1["tipo"], orientation="h", width=0.5,
    marker=dict(color=[COLOR_TIPO[t] for t in g1["tipo"]], cornerradius=4),
    text=[f"{n} · {p:.0f} %" for n, p in zip(g1["n"], g1["pct"])], textposition="outside",
    textfont=dict(color=TINTA2), cliponaxis=False,
    customdata=np.c_[g1["mixtos"], g1["ids"]],
    hovertemplate="<b>%{y}</b>: %{x} documentos<br>mixtos: %{customdata[0]}<br>%{customdata[1]}<extra></extra>"))
fig.update_layout(
    title=dict(text=f"Corpus documental equilibrado: {g1.loc[0, 'n']} impresos, {g1.loc[1, 'n']} fotos y {g1.loc[2, 'n']} manuscritos",
               subtitle=dict(text=f"{len(clasif)} documentos; {int(clasif['es_mixto'].sum())} son mixtos (formulario o reproducción impresa con contenido manuscrito)")),
    xaxis=dict(title="Número de documentos", range=[0, g1["n"].max() * 1.25], dtick=2),
    yaxis=dict(categoryorder="array", categoryarray=ORDEN_TIPO[::-1], title=None), height=330, showlegend=False)
FIGURAS["g1_documentos_por_tipo"] = fig
fig.show()

In [18]:
# Gráfico 2 — Duración real del audio por fuente (cada segmento es un clip)
orden_fuentes = (clips.groupby("fuente_etiqueta")["duracion_vivo_s"].sum().sort_values()).index.tolist()
tot = clips.groupby("fuente_etiqueta")["duracion_vivo_s"].sum() / 60
top2 = tot.sort_values(ascending=False).head(2)
fig = go.Figure()
for _, c in clips.sort_values(["fuente_id", "inicio_s"]).iterrows():
    fig.add_trace(go.Bar(
        x=[c["duracion_vivo_s"] / 60], y=[c["fuente_etiqueta"]], orientation="h", width=0.55,
        marker=dict(color=CATEGORICA[0], line=dict(color=SUPERFICIE, width=2)), showlegend=False,
        customdata=[[c["id"], c["duracion_vivo_s"], c["inicio_s"]]],
        hovertemplate="<b>%{customdata[0]}</b> · %{y}<br>duración: %{customdata[1]:.1f} s"
                      "<br>inicio en la fuente: %{customdata[2]:.0f} s<extra></extra>"))
for f, m in tot.items():
    fig.add_annotation(x=m, y=f, text=f"{m:.1f} min · {int((clips['fuente_etiqueta'] == f).sum())} clips",
                       showarrow=False, xanchor="left", xshift=8, font=dict(color=TINTA2, size=12))
total_min = clips["duracion_vivo_s"].sum() / 60
fig.update_layout(
    barmode="stack", height=430,
    title=dict(text=f"{total_min:.1f} min reales en {len(clips)} clips: {top2.index[0][0]} y {top2.index[1][0]} aportan el "
                    f"{100 * top2.sum() / total_min:.0f} % del audio",
               subtitle=dict(text="Duración medida con ffprobe; cada segmento de la barra es un clip WAV mono de 16 kHz cortado en una pausa")),
    xaxis=dict(title="Minutos", range=[0, tot.max() * 1.35]),
    yaxis=dict(categoryorder="array", categoryarray=orden_fuentes, title=None))
FIGURAS["g2_duracion_audio"] = fig
fig.show()

In [19]:
# Gráfico 3 — Longitud del texto extraído: caracteres OCR por documento y palabras ASR por minuto
med_tipo = ocr_df.groupby("tipo")["n_caracteres"].median().reindex(ORDEN_TIPO)
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.14,
                    subplot_titles=("Caracteres OCR por documento", "Palabras ASR por minuto de audio"))
rng = np.random.default_rng(7)  # dispersión vertical fija (solo visual) para no superponer puntos
for t in ORDEN_TIPO:
    d = ocr_df[ocr_df["tipo"] == t]
    fig.add_trace(go.Box(x=d["n_caracteres"], y=[t] * len(d), orientation="h", name=t, boxpoints=False,
                         line=dict(color=COLOR_TIPO[t], width=1.5), fillcolor="rgba(0,0,0,0)", hoverinfo="skip",
                         showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=d["n_caracteres"], y=[t] * len(d), mode="markers", name=t, showlegend=False,
                             marker=dict(size=10, color=COLOR_TIPO[t], line=dict(color=SUPERFICIE, width=2)),
                             customdata=np.c_[d["id"], d["confianza_media"]],
                             hovertemplate="<b>%{customdata[0]}</b> (%{y})<br>%{x} caracteres<br>confianza %{customdata[1]:.2f}<extra></extra>"), 1, 1)
orden_f = sorted(asr_df["fuente_id"].unique())
fig.add_trace(go.Box(x=asr_df["palabras_por_min"], y=asr_df["fuente_id"].str[-1], orientation="h", boxpoints=False,
                     line=dict(color=TENUE, width=1.5), fillcolor="rgba(0,0,0,0)", hoverinfo="skip", showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=asr_df["palabras_por_min"], y=asr_df["fuente_id"].str[-1], mode="markers", showlegend=False,
                         marker=dict(size=10, color=CATEGORICA[0], line=dict(color=SUPERFICIE, width=2)),
                         customdata=np.c_[asr_df["id"], asr_df["n_palabras"], asr_df["duracion_seg"]],
                         hovertemplate="<b>%{customdata[0]}</b> · fuente %{y}<br>%{x:.0f} palabras/min"
                                       "<br>%{customdata[1]} palabras en %{customdata[2]:.0f} s<extra></extra>"), 1, 2)
mayor, menor = med_tipo.idxmax(), med_tipo.idxmin()
fig.update_layout(
    height=470,
    title=dict(text=f"El OCR de {mayor}s produce la mediana más alta de caracteres ({med_tipo[mayor]:.0f}) y el de {menor}s la más baja ({med_tipo[menor]:.0f})",
               subtitle=dict(text=f"ASR: mediana de {asr_df['palabras_por_min'].median():.0f} palabras por minuto en {len(asr_df)} clips; "
                                  "dos medidas distintas en paneles separados (sin doble eje)")),
    margin=dict(t=120))
fig.update_xaxes(title_text="Caracteres (escala logarítmica)", type="log", row=1, col=1,
                 tickvals=[100, 200, 500, 1000, 2000, 5000, 10000],
                 ticktext=["100", "200", "500", "1 000", "2 000", "5 000", "10 000"])
fig.update_xaxes(title_text="Palabras por minuto", row=1, col=2)
fig.update_yaxes(categoryorder="array", categoryarray=ORDEN_TIPO[::-1], row=1, col=1)
fig.update_yaxes(categoryorder="array", categoryarray=[f[-1] for f in orden_f][::-1], title_text="Fuente", row=1, col=2)
fig.update_annotations(font=dict(size=13, color=TINTA2))
FIGURAS["g3_longitud_texto"] = fig
fig.show()

In [20]:
# Gráfico 4 — Confianza media de PaddleOCR por tipo (señal automática, no exactitud: el CER se mide en la Parte 2)
med_conf = ocr_df.groupby("tipo")["confianza_media"].median().reindex(ORDEN_TIPO)
fig = go.Figure()
for t in ORDEN_TIPO:
    d = ocr_df[ocr_df["tipo"] == t].sort_values("confianza_media")
    fig.add_trace(go.Scatter(x=d["confianza_media"], y=[t] * len(d), mode="markers", name=t, showlegend=False,
                             marker=dict(size=11, color=COLOR_TIPO[t], line=dict(color=SUPERFICIE, width=2)),
                             customdata=np.c_[d["id"], d["n_lineas"]],
                             hovertemplate="<b>%{customdata[0]}</b> (%{y})<br>confianza media %{x:.3f}<br>%{customdata[1]} líneas<extra></extra>"))
    fig.add_trace(go.Scatter(x=[med_conf[t]] * 2, y=[t, t], mode="markers", marker=dict(symbol="line-ns", size=26,
                             line=dict(width=2, color=TINTA)), showlegend=False,
                             hovertemplate=f"mediana {t}: {med_conf[t]:.3f}<extra></extra>"))
fig.update_layout(
    title=dict(text=f"Confianza de PaddleOCR: mediana {med_conf['impreso']:.2f} en impresos frente a {med_conf['manuscrito']:.2f} en manuscritos",
               subtitle=dict(text="Cada punto es un documento; la marca negra es la mediana del tipo. La confianza es autoinformada por el modelo, no exactitud")),
    xaxis=dict(title="Confianza media de reconocimiento por línea", range=[0.4, 1.0]),
    yaxis=dict(categoryorder="array", categoryarray=ORDEN_TIPO[::-1], title=None), height=340)
FIGURAS["g4_confianza_ocr"] = fig
fig.show()

**Lectura de los gráficos.**
1. **Tipos de documento:** la distribución cumple el diseño (8/8/9).
2. **Duración del audio:** está concentrada. F y A aportan el 45 % de los minutos, mientras que C solo suma 2,8 min en 2 clips; esto debe considerarse al formar la muestra de evaluación.
3. **Longitud del texto:** los manuscritos producen menos texto OCR (mediana de 444 caracteres frente a 1870 en impresos). Un texto breve no implica necesariamente un error, porque hay documentos breves como los recibos. Sin embargo, el poco volumen sumado a la menor confianza sugiere pérdidas que la Parte 2 debe medir. En el audio, el ritmo de habla va de una mediana de 117 palabras/min (E, ceremonia con música) a 181 (C).
4. **Confianza del OCR:** es alta y homogénea en impresos (mediana 0,97), dispersa en fotos (de 0,72 a 0,99) y más baja en manuscritos (mediana 0,81). Es una señal, no una medida de exactitud: `doc_003` (manuscrito) tiene una confianza de 0,94 pero solo 113 caracteres.

### Exportación estática (opcional)
Los gráficos de arriba son **interactivos** (pase el cursor para ver los valores) y se visualizan en Jupyter, VS Code y nbviewer. Si se necesitan imágenes PNG (por ejemplo, para un informe), se activa `EXPORTAR_PNG = True`: cada figura se exporta en un **subproceso con un límite de 60 s**. Si Kaleido no responde, se informa y la ejecución continúa sin bloquearse.

In [21]:
crono.marca("8b. Exportación PNG opcional")
if EXPORTAR_PNG:
    for nombre, fig in FIGURAS.items():
        print(graficos.exportar_png(fig, DATOS / "ejemplos" / f"{nombre}.png", limite_s=60))
else:
    print(f"Exportación PNG desactivada (EXPORTAR_PNG = False); {len(FIGURAS)} figuras disponibles en modo interactivo.")

Exportación PNG desactivada (EXPORTAR_PNG = False); 6 figuras disponibles en modo interactivo.


### Extremos del corpus

In [22]:
crono.marca("9. Extremos del corpus")
cols_ocr = ["id", "tipo", "n_caracteres", "n_lineas", "confianza_media"]
largos = ocr_df.nlargest(3, "n_caracteres")[cols_ocr].assign(extremo="OCR más largo")
cortos = ocr_df.nsmallest(3, "n_caracteres")[cols_ocr].assign(extremo="OCR más corto")
display(pd.concat([largos, cortos]).style.hide(axis="index").format({"confianza_media": "{:.3f}"}))
cols_a = ["id", "fuente_etiqueta", "duracion_seg", "n_palabras", "pct_voz_vad", "n_hablantes_estimado"]
a_largos = asr_df.nlargest(3, "duracion_seg")[cols_a].assign(extremo="audio más largo")
a_cortos = asr_df.nsmallest(3, "duracion_seg")[cols_a].assign(extremo="audio más corto")
display(pd.concat([a_largos, a_cortos]).style.hide(axis="index").format({"duracion_seg": "{:.1f}", "pct_voz_vad": "{:.0f} %"}))
for i in list(largos["id"][:1]) + list(cortos["id"][:1]):
    t = ocr_df.set_index("id").loc[i, "texto_ocr_seud"]
    display(Markdown(f"**{i}** — primeras líneas del OCR (seudonimizado)\n\n```text\n" + "\n".join(t.splitlines()[:6]) + "\n```"))

id,tipo,n_caracteres,n_lineas,confianza_media,extremo
doc_025,impreso,11468,1376,0.939,OCR más largo
doc_022,impreso,2374,38,0.994,OCR más largo
doc_007,impreso,2329,80,0.978,OCR más largo
doc_003,manuscrito,113,9,0.936,OCR más corto
doc_002,manuscrito,232,17,0.848,OCR más corto
doc_005,impreso,238,11,0.972,OCR más corto


id,fuente_etiqueta,duracion_seg,n_palabras,pct_voz_vad,n_hablantes_estimado,extremo
aud_011,C · Entrevista en Plaza San Martín (2),88.9,262,100 %,1,audio más largo
aud_019,E · Entrevistas en ceremonia patronal,84.5,107,84 %,2,audio más largo
aud_015,D · Excombatiente del Cenepa,82.5,237,100 %,1,audio más largo
aud_029,G · Vendedoras en El Agustino,58.0,119,97 %,2,audio más corto
aud_030,G · Vendedoras en El Agustino,64.9,113,95 %,2,audio más corto
aud_009,B · Entrevista en Plaza San Martín (1),64.9,136,99 %,1,audio más corto


**doc_025** — primeras líneas del OCR (seudonimizado)

```text
80
78
81
79
77
76
```

**doc_003** — primeras líneas del OCR (seudonimizado)

```text
ral & South American Telegraph Co.
RECIBO.
Para, parklite Ofevele
1915
487/45
Ralabras,
```

**Observación: extremos.**
- **OCR más largos:** `doc_025`, con 11 468 caracteres (es un mapa con miles de rótulos), seguido de `doc_022` y `doc_007` (impresos densos de texto corrido).
- **OCR más cortos:** `doc_003` (113 caracteres) y `doc_002` (232), que son recibos con poco texto y escritura a mano, y `doc_005` (238), una portada con pocas palabras. Un OCR corto puede deberse a un documento breve (la portada) o a una lectura fallida (el manuscrito); solo la referencia humana permite distinguirlos.
- **Audios:** por diseño, todos los clips duran entre 58 y 89 s (se cortaron en pausas). `aud_019` (E) tiene 107 palabras en 84,6 s, frente a las 262 de `aud_011` (C) en 88,9 s: la música y los silencios de la ceremonia reducen el texto transcrito.

## 9. Estado de la Parte 1 y pendientes de revisión humana

In [23]:
crono.marca("10. Resumen de pendientes")
pend_tabla = pd.DataFrame([
    ("Idioma final de F y G (escucha)", f"{(tr_out['idioma'] == PENDIENTE).sum()} clips", "Escuchar al menos 30 s por clip de F (aud_021–027) y G (aud_028–030)"),
    ("Número de hablantes (n_hablantes)", f"{(tr_out['n_hablantes'] == PENDIENTE).sum()} clips",
     f"Escuchar el caso multihablante ({caso}) y confirmar o corregir las etiquetas [HABLANTE_k]"),
    ("Fuentes de documentos sin verificar", f"{(fuentes_out['licencia'] == PENDIENTE).sum()} documentos", "Aportar URL y licencia de " + ", ".join(pend)),
    ("Revisión de seudonimización", f"{len(docs_out) + len(tr_out)} textos", "Revisar reemplazos y nombres no detectados; confirmar la lista de personas públicas"),
    ("Referencias manuales (texto_referencia)", f"{(docs_out['texto_referencia'] == PENDIENTE).sum() + (tr_out['texto_referencia'] == PENDIENTE).sum()} textos",
     "Se completan en la Parte 2 solo para la muestra fija (10 documentos + 10 clips)"),
], columns=["Pendiente", "Alcance", "Acción humana"])
display(pend_tabla.style.hide(axis="index"))

Pendiente,Alcance,Acción humana
Idioma final de F y G (escucha),10 clips,Escuchar al menos 30 s por clip de F (aud_021–027) y G (aud_028–030)
Número de hablantes (n_hablantes),30 clips,Escuchar el caso multihablante (audio_F) y confirmar o corregir las etiquetas [HABLANTE_k]
Fuentes de documentos sin verificar,6 documentos,"Aportar URL y licencia de doc_005.png, doc_006.png, doc_007.png, doc_009.png, doc_010.png, doc_021.jpg"
Revisión de seudonimización,55 textos,Revisar reemplazos y nombres no detectados; confirmar la lista de personas públicas
Referencias manuales (texto_referencia),55 textos,Se completan en la Parte 2 solo para la muestra fija (10 documentos + 10 clips)


**Estado.** La Parte 1 queda ejecutada de punta a punta con resultados reales: OCR de 25 documentos, ASR + VAD de 30 clips, diarización de 7 fuentes, seudonimización con verificación automática y exportación de los CSV. Todo lo que depende de juicio humano está marcado como `PENDIENTE_REVISION_HUMANA` en la tabla anterior.

### Tiempos de ejecución de este notebook
Los procesos pesados se leen de la caché: el OCR tomó 10,2 h y el ASR 16,4 min en su ejecución original, con tiempos registrados por elemento. Esta tabla mide **solo esta ejecución del notebook** y se guarda en `datos/reportes/tiempos_notebook_01.csv`.

In [24]:
crono.marca(None)
tiempos = pd.DataFrame(crono.registros)
tiempos.to_csv(REPORTES / "tiempos_notebook_01.csv", index=False, encoding="utf-8")
display(tiempos.style.hide(axis="index").format({"segundos": "{:.1f}"}))
print(f"Total: {tiempos['segundos'].sum():.1f} s")

seccion,segundos
0. Configuración e importaciones,1.2
1. Carga y validación del corpus (hashes y ffprobe en vivo),1.3
2. OCR PaddleOCR (lectura de caché si existe),2.2
3. ASR faster-whisper (lectura de caché si existe),0.1
4a. Diarización (lectura de caché si existe) y alineación con ASR,0.0
4b. Gráficos de diarización,0.2
5. Seudonimización (dos pasadas + verificación de fugas),2.9
6. Duplicados y descartes (dHash + VAD),1.1
7. Escritura de CSV finales,0.1
8. Gráficos EDA (interactivos),0.2


Total: 9.4 s
